In [1]:
# ============================================================================
# REGENERATE PREDICTIONS & CREDIT LIMITS FOR ALL OBSERVATION DATES
# Fix: Score ALL retailers across ALL dates, not just test set
# ============================================================================

from pyspark.ml import PipelineModel
from pyspark.sql import functions as F

print("=" * 80)
print("REGENERATING PREDICTIONS FOR ALL OBSERVATION DATES")
print("=" * 80)

# ============================================================================
# STEP 1: LOAD THE TRAINED MODEL
# ============================================================================

print("\n1. Loading trained model...")

# Load your GBT model from MLflow or saved location
# Option A: Load from MLflow
try:
    import mlflow
    model_uri = "models:/credit_score_gbt_v1/latest"  # Adjust if needed
    gbt_model = mlflow.spark.load_model(model_uri)
    print("✓ Loaded model from MLflow")
except:
    # Option B: Load from table/path if you saved it
    print("⚠️ Could not load from MLflow, trying alternate method...")
    # You may need to retrain if model wasn't saved properly

# ============================================================================
# STEP 2: LOAD ALL FEATURES (8 OBSERVATION DATES)
# ============================================================================

print("\n2. Loading feature data for ALL observation dates...")

all_features = spark.table("gold_credit_scoring_features")

print(f"✓ Loaded {all_features.count():,} records")
print(f"✓ Spanning {all_features.select('observation_date').distinct().count()} observation dates")

all_features.groupBy("observation_date").count().orderBy("observation_date").show()

# ============================================================================
# STEP 3: GENERATE PREDICTIONS FOR ALL DATES
# ============================================================================

print("\n3. Generating predictions for ALL dates...")

# Apply model to full dataset
try:
    all_predictions = gbt_model.transform(all_features)
    
    # Assign predicted risk tiers
    all_predictions = all_predictions.withColumn(
        "predicted_tier",
        F.when(F.col("prediction") >= 750, "Platinum")
         .when(F.col("prediction") >= 650, "Gold")
         .when(F.col("prediction") >= 550, "Silver")
         .when(F.col("prediction") >= 450, "Bronze")
         .otherwise("Copper")
    )
    
    all_predictions = all_predictions.withColumn(
        "predicted_decision",
        F.when(F.col("predicted_tier").isin(["Platinum", "Gold"]), "APPROVE")
         .when(F.col("predicted_tier") == "Silver", "APPROVE_WITH_MONITORING")
         .when(F.col("predicted_tier") == "Bronze", "CONDITIONAL")
         .otherwise("DECLINE")
    )
    
    # Select columns matching your existing predictions table
    predictions_full = all_predictions.select(
        "retailer_id",
        "observation_date",
        "credit_score",
        F.col("prediction").alias("predicted_score"),
        "risk_tier",
        "predicted_tier",
        "predicted_decision",
        "moderate_default",
        "serious_default",
    )
    
    print(f"✓ Generated {predictions_full.count():,} predictions")
    
    # Save to predictions table
    predictions_full.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("gold_credit_score_predictions")
    
    print("✓ Saved to gold_credit_score_predictions")
    
except Exception as e:
    print(f"\n❌ ERROR generating predictions: {str(e)}")
    print("\n⚠️ You may need to retrain the model on the full dataset")
    print("   See alternative solution below...")

# ============================================================================
# ALTERNATIVE: RETRAIN MODEL ON FULL DATASET
# ============================================================================

print("\n" + "=" * 80)
print("ALTERNATIVE: QUICK RETRAIN ON FULL DATASET")
print("=" * 80)

# If model loading failed, do a quick retrain
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline

print("\nRetraining model on ALL data (no train/test split)...")

# Feature columns (same as before)
feature_cols = [
    "on_time_rate_recent", "on_time_rate_medium", "on_time_rate_lifetime",
    "avg_days_late_recent", "avg_days_late_medium", "avg_days_late_lifetime",
    "max_days_late_recent", "max_days_late_lifetime",
    "late_rate_recent", "late_rate_medium", "serious_late_rate_recent",
    "payment_consistency", "stddev_days_late_recent",
    "txn_count_recent", "txn_count_medium", "txn_count_lifetime",
    "avg_orders_per_month", "days_since_last_order", "customer_tenure_days",
    "unique_categories",
    "avg_order_value_recent", "avg_order_value_medium", "avg_order_value_lifetime",
    "total_value_recent", "total_value_lifetime",
    "credit_utilization", "stddev_order_value",
    "payment_deterioration_ratio", "late_rate_change",
    "txn_velocity_ratio", "on_time_improvement",
    "formality_score", "mobile_money_score",
    "owner_age", "years_in_business", "num_employees",
    "gender_encoded", "urbanization_encoded", "shop_type_encoded",
    "ever_seriously_late_recent", "ever_defaulted_lifetime",
    "usd_ngn_rate", "inflation_rate_pct", "petrol_price_ngn",
]

# Prepare data
model_data = all_features.select(
    ["retailer_id", "observation_date", "credit_score", "risk_tier"] + feature_cols
).na.drop(subset=["credit_score"])

print(f"Training on {model_data.count():,} samples...")

# Build pipeline
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

gbt = GBTRegressor(
    labelCol="credit_score",
    featuresCol="features",
    predictionCol="predicted_score",
    maxIter=100,
    maxDepth=6,
    stepSize=0.1,
    subsamplingRate=0.8,
    seed=42
)

pipeline = Pipeline(stages=[assembler, scaler, gbt])

# Train on full dataset
model_full = pipeline.fit(model_data)

print("✓ Model retrained")

# Generate predictions
predictions_full = model_full.transform(model_data)

# Assign tiers
predictions_full = predictions_full.withColumn(
    "predicted_tier",
    F.when(F.col("predicted_score") >= 750, "Platinum")
     .when(F.col("predicted_score") >= 650, "Gold")
     .when(F.col("predicted_score") >= 550, "Silver")
     .when(F.col("predicted_score") >= 450, "Bronze")
     .otherwise("Copper")
)

# Select final columns
predictions_full = predictions_full.select(
    "retailer_id",
    "observation_date",
    "credit_score",
    "predicted_score",
    "risk_tier",
    "predicted_tier",
    F.when(F.col("predicted_tier").isin(["Platinum", "Gold"]), "APPROVE")
     .when(F.col("predicted_tier") == "Silver", "APPROVE_WITH_MONITORING")
     .when(F.col("predicted_tier") == "Bronze", "CONDITIONAL")
     .otherwise("DECLINE").alias("predicted_decision"),
)

# Add default columns (join from features)
predictions_full = predictions_full.join(
    all_features.select("retailer_id", "observation_date", 
                       "moderate_default", "serious_default"),
    ["retailer_id", "observation_date"],
    "left"
)

# Save
predictions_full.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_credit_score_predictions")

print(f"✓ Saved {predictions_full.count():,} predictions to gold_credit_score_predictions")

# Verify dates
print("\nPredictions by date:")
predictions_full.groupBy("observation_date").agg(
    F.count("*").alias("records"),
    F.avg("predicted_score").alias("avg_score")
).orderBy("observation_date").show()

# ============================================================================
# STEP 4: REGENERATE CREDIT LIMITS FOR ALL DATES
# ============================================================================

print("\n" + "=" * 80)
print("4. REGENERATING CREDIT LIMITS FOR ALL DATES")
print("=" * 80)

# Load fresh predictions
predictions_all = spark.table("gold_credit_score_predictions")
features_all = spark.table("gold_credit_scoring_features")

# Join predictions with features
credit_data = predictions_all.join(
    features_all.select(
        "retailer_id", "observation_date",
        "avg_order_value_lifetime", "total_value_lifetime",
        "txn_count_lifetime", "credit_limit",
        "avg_orders_per_month", "on_time_rate_lifetime",
        "shop_type", "urbanization_level", "years_in_business",
    ),
    ["retailer_id", "observation_date"],
    "left"
)

print(f"Processing {credit_data.count():,} records...")

# Calculate multipliers (same logic as before)
credit_data = credit_data.withColumn(
    "base_credit_limit",
    F.when(F.col("predicted_tier") == "Platinum", 500000)
     .when(F.col("predicted_tier") == "Gold", 250000)
     .when(F.col("predicted_tier") == "Silver", 100000)
     .when(F.col("predicted_tier") == "Bronze", 50000)
     .otherwise(0)
)

credit_data = credit_data.withColumn(
    "history_multiplier",
    F.when(F.col("txn_count_lifetime") >= 100, 2.0)
     .when(F.col("txn_count_lifetime") >= 50, 1.8)
     .when(F.col("txn_count_lifetime") >= 30, 1.5)
     .when(F.col("txn_count_lifetime") >= 15, 1.3)
     .when(F.col("txn_count_lifetime") >= 5, 1.1)
     .otherwise(1.0)
)

credit_data = credit_data.withColumn(
    "payment_bonus",
    F.when(F.col("on_time_rate_lifetime") >= 0.99, 1.5)
     .when(F.col("on_time_rate_lifetime") >= 0.95, 1.3)
     .when(F.col("on_time_rate_lifetime") >= 0.90, 1.2)
     .when(F.col("on_time_rate_lifetime") >= 0.85, 1.1)
     .otherwise(1.0)
)

credit_data = credit_data.withColumn(
    "maturity_bonus",
    F.when(F.col("years_in_business") >= 10, 1.3)
     .when(F.col("years_in_business") >= 5, 1.2)
     .when(F.col("years_in_business") >= 3, 1.1)
     .otherwise(1.0)
)

credit_data = credit_data.withColumn(
    "shop_type_multiplier",
    F.when(F.col("shop_type") == "Superette", 1.2)
     .when(F.col("shop_type") == "Mini Mart", 1.1)
     .when(F.col("shop_type") == "Provision Store", 1.0)
     .when(F.col("shop_type") == "Market Stall", 0.9)
     .otherwise(0.8)
)

credit_data = credit_data.withColumn(
    "location_multiplier",
    F.when(F.col("urbanization_level") == "Urban", 1.2)
     .when(F.col("urbanization_level") == "Peri-Urban", 1.0)
     .otherwise(0.9)
)

credit_data = credit_data.withColumn(
    "total_multiplier",
    F.col("history_multiplier") * F.col("payment_bonus") * 
    F.col("maturity_bonus") * F.col("shop_type_multiplier") * 
    F.col("location_multiplier")
)

credit_data = credit_data.withColumn(
    "calculated_limit",
    (F.col("base_credit_limit") * F.col("total_multiplier")).cast("long")
)

credit_data = credit_data.withColumn(
    "final_credit_limit",
    F.when(F.col("predicted_tier") == "Platinum", 
           F.least(F.col("calculated_limit"), F.lit(2000000)))
     .when(F.col("predicted_tier") == "Gold", 
           F.least(F.col("calculated_limit"), F.lit(1000000)))
     .when(F.col("predicted_tier") == "Silver", 
           F.least(F.col("calculated_limit"), F.lit(500000)))
     .when(F.col("predicted_tier") == "Bronze", 
           F.least(F.col("calculated_limit"), F.lit(150000)))
     .otherwise(0)
)

credit_data = credit_data.withColumn(
    "expected_default_rate",
    F.when(F.col("predicted_tier") == "Platinum", 0.001)
     .when(F.col("predicted_tier") == "Gold", 0.005)
     .when(F.col("predicted_tier") == "Silver", 0.01)
     .when(F.col("predicted_tier") == "Bronze", 0.03)
     .otherwise(0.30)
)

credit_data = credit_data.withColumn(
    "expected_loss",
    (F.col("final_credit_limit") * F.col("expected_default_rate")).cast("long")
)

credit_data = credit_data.withColumn(
    "recommended_utilization",
    F.col("avg_order_value_lifetime") / F.col("final_credit_limit")
)

credit_data = credit_data.withColumn(
    "credit_decision_detail",
    F.when(F.col("predicted_tier") == "Platinum", 
           F.concat(F.lit("APPROVE - ₦"), F.format_number(F.col("final_credit_limit"), 0)))
     .when(F.col("predicted_tier") == "Gold", 
           F.concat(F.lit("APPROVE - ₦"), F.format_number(F.col("final_credit_limit"), 0)))
     .when(F.col("predicted_tier") == "Silver", 
           F.concat(F.lit("APPROVE (Monitor) - ₦"), F.format_number(F.col("final_credit_limit"), 0)))
     .when(F.col("predicted_tier") == "Bronze", 
           F.concat(F.lit("CONDITIONAL - ₦"), F.format_number(F.col("final_credit_limit"), 0)))
     .otherwise("DECLINE - No Credit")
)

# Select final columns
credit_limits_full = credit_data.select(
    "retailer_id",
    "observation_date",
    "predicted_score",
    "predicted_tier",
    "base_credit_limit",
    "final_credit_limit",
    "credit_decision_detail",
    "total_multiplier",
    "history_multiplier",
    "payment_bonus",
    "maturity_bonus",
    "shop_type_multiplier",
    "location_multiplier",
    "expected_default_rate",
    "expected_loss",
    "recommended_utilization",
)

# Add moderate_default from predictions
credit_limits_full = credit_limits_full.join(
    predictions_all.select("retailer_id", "observation_date", "moderate_default"),
    ["retailer_id", "observation_date"],
    "left"
)

# Save
credit_limits_full.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_credit_limits")

print(f"✓ Saved {credit_limits_full.count():,} credit limit records")

print("\nCredit limits by date:")
credit_limits_full.groupBy("observation_date").agg(
    F.count("*").alias("records"),
    F.sum("final_credit_limit").alias("total_exposure")
).orderBy("observation_date").show()

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("✅ REGENERATION COMPLETE")
print("=" * 80)

pred_count = spark.table("gold_credit_score_predictions").count()
pred_dates = spark.table("gold_credit_score_predictions").select("observation_date").distinct().count()
limits_count = spark.table("gold_credit_limits").count()
limits_dates = spark.table("gold_credit_limits").select("observation_date").distinct().count()

print(f"\nUpdated tables:")
print(f"  gold_credit_score_predictions:")
print(f"    Records: {pred_count:,}")
print(f"    Observation dates: {pred_dates}")
print(f"\n  gold_credit_limits:")
print(f"    Records: {limits_count:,}")
print(f"    Observation dates: {limits_dates}")

if pred_dates >= 8 and limits_dates >= 8:
    print("\n✓ SUCCESS: Both tables now contain all observation dates!")
    print("\nNext step: Rebuild fact table using the 'rebuild_fact_table_full' script")
else:
    print(f"\n⚠️ WARNING: Expected 8 dates, got {pred_dates} predictions and {limits_dates} limits")

print("=" * 80)

StatementMeta(, ace30b88-f8dc-4d9c-8595-f2ea91c9e423, 3, Finished, Available, Finished)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/mlflow/store/artifact/utils/models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.12.2/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])
2026-02-03:08:24:41,357 ERROR    [synapse_mlflow_utils.py:383] [fabric mlflow plugin]: <class 'mlflow.store.model_registry.rest_store.RestStore'>.get_latest_versions exception PERMISSION_DENIED: Response: {'Message': 'User does not have permission to perform this operation.', 'Source': 'ML', 'error_code': 'PERMISSION_DENIED'}


⚠️ Could not load from MLflow, trying alternate method...

2. Loading feature data for ALL observation dates...
✓ Loaded 8,150 records
✓ Spanning 8 observation dates
+----------------+-----+
|observation_date|count|
+----------------+-----+
|      2024-07-25|  102|
|      2024-08-01|  249|
|      2024-08-10|  449|
|      2024-08-20|  700|
|      2024-09-01| 1023|
|      2024-09-15| 1424|
|      2024-09-30| 1876|
|      2024-10-15| 2327|
+----------------+-----+


3. Generating predictions for ALL dates...

❌ ERROR generating predictions: name 'gbt_model' is not defined

⚠️ You may need to retrain the model on the full dataset
   See alternative solution below...

ALTERNATIVE: QUICK RETRAIN ON FULL DATASET

Retraining model on ALL data (no train/test split)...
Training on 8,150 samples...


✓ Model retrained
✓ Saved 6,266 predictions to gold_credit_score_predictions

Predictions by date:
+----------------+-------+-----------------+
|observation_date|records|        avg_score|
+----------------+-------+-----------------+
|      2024-07-25|     32|600.2950312205472|
|      2024-08-01|    104|597.7283234050419|
|      2024-08-10|    251|601.2649402687462|
|      2024-08-20|    468|592.6550176557898|
|      2024-09-01|    743|595.6299817760743|
|      2024-09-15|   1122|593.9927983426437|
|      2024-09-30|   1540|592.3802984332871|
|      2024-10-15|   2006|592.7553622621458|
+----------------+-------+-----------------+


4. REGENERATING CREDIT LIMITS FOR ALL DATES
Processing 6,266 records...
✓ Saved 6,266 credit limit records

Credit limits by date:
+----------------+-------+--------------+
|observation_date|records|total_exposure|
+----------------+-------+--------------+
|      2024-07-25|     32|       4773799|
|      2024-08-01|    104|      14295954|
|      2024-08-10|